In [1]:
import pandas as pd
import numpy as np

import pickle
import gzip
import json

from math import log, exp

from huggingface_hub import hf_hub_download

In [2]:
REPO_ID = "rajdeeppa53/hindi-ngram-language-models"

print(
    "Using Hugging Face repository:",
    REPO_ID
)

Using Hugging Face repository: rajdeeppa53/hindi-ngram-language-models


#### Function to load models

In [3]:
def load_pickle_from_huggingface(
    repo_id,
    filename
):
    
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type="model"
    )
    
    with gzip.open(
        local_path,
        "rb"
    ) as f:
        
        return pickle.load(f)

### Download models from HF

In [4]:
vocab = load_pickle_from_huggingface(
    REPO_ID,
    "vocab.pkl.gz"
)

V = len(vocab)

print("Vocabulary size:", V)

vocab.pkl.gz:   0%|          | 0.00/3.80M [00:00<?, ?B/s]

Vocabulary size: 545854


c:\Users\rajde\anaconda3\envs\pytrch310\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rajde\.cache\huggingface\hub\models--rajdeeppa53--hindi-ngram-language-models. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [5]:
unigram_counts = load_pickle_from_huggingface(
    REPO_ID,
    "unigram_counts.pkl.gz"
)

print(
    "Unique unigrams:",
    len(unigram_counts)
)

unigram_counts.pkl.gz:   0%|          | 0.00/4.17M [00:00<?, ?B/s]

Unique unigrams: 545853


In [6]:
bigram_counts = load_pickle_from_huggingface(
    REPO_ID,
    "bigram_counts.pkl.gz"
)

bigram_context_counts = load_pickle_from_huggingface(
    REPO_ID,
    "bigram_context_counts.pkl.gz"
)

print(
    "Unique bigrams:",
    len(bigram_counts)
)

bigram_counts.pkl.gz:   0%|          | 0.00/48.9M [00:00<?, ?B/s]

bigram_context_counts.pkl.gz:   0%|          | 0.00/4.17M [00:00<?, ?B/s]

Unique bigrams: 7723522


In [7]:
trigram_counts = load_pickle_from_huggingface(
    REPO_ID,
    "trigram_counts.pkl.gz"
)

trigram_context_counts = load_pickle_from_huggingface(
    REPO_ID,
    "trigram_context_counts.pkl.gz"
)

print(
    "Unique trigrams:",
    len(trigram_counts)
)

trigram_counts.pkl.gz:   0%|          | 0.00/156M [00:00<?, ?B/s]

trigram_context_counts.pkl.gz:   0%|          | 0.00/48.7M [00:00<?, ?B/s]

Unique trigrams: 26029580


In [8]:
quadrigram_counts = load_pickle_from_huggingface(
    REPO_ID,
    "quadrigram_counts.pkl.gz"
)

quadrigram_context_counts = load_pickle_from_huggingface(
    REPO_ID,
    "quadrigram_context_counts.pkl.gz"
)

print(
    "Unique quadrigrams:",
    len(quadrigram_counts)
)

quadrigram_counts.pkl.gz:   0%|          | 0.00/238M [00:00<?, ?B/s]

quadrigram_context_counts.pkl.gz:   0%|          | 0.00/155M [00:00<?, ?B/s]

Unique quadrigrams: 42140419


In [9]:
metadata_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="metadata.json",
    repo_type="model"
)

with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as f:
    
    metadata = json.load(f)

print(
    json.dumps(
        metadata,
        indent=4,
        ensure_ascii=False
    )
)

metadata.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

{
    "language": "Hindi",
    "model_type": "Count-based N-gram Language Models",
    "models": [
        "unigram",
        "bigram",
        "trigram",
        "quadrigram"
    ],
    "vocabulary_size": 545854,
    "total_tokens": 72441305,
    "start_token": "<s>",
    "end_token": "</s>",
    "unk_token": "<UNK>",
    "training_sentences": 3033640,
    "development_sentences": 1000,
    "test_sentences": 1000,
    "smoothing": "Applied during probability calculation",
    "previous_experiment": "Add-One/Laplace",
    "next_experiment": "Add-K"
}


### Split into Test and Development sets using the same seed

In [10]:
df = pd.read_parquet(
    "../Assignment 01/hindi_texts_tokenized.parquet"
)

In [11]:
SEED = 42

rng = np.random.default_rng(SEED)

N = len(df)

selected_indices = rng.choice(
    N,
    size=2000,
    replace=False
)

dev_indices = selected_indices[:1000]

test_indices = selected_indices[1000:]

dev_df = df.iloc[dev_indices]

test_df = df.iloc[test_indices]

print("Development:", len(dev_df))
print("Test:", len(test_df))

Development: 1000
Test: 1000


In [12]:
K = 0.3

print(
    "Add-K smoothing parameter:",
    K
)

Add-K smoothing parameter: 0.3


In [13]:
START_TOKEN = metadata["start_token"]
END_TOKEN = metadata["end_token"]
UNK_TOKEN = metadata["unk_token"]
def add_sentence_tokens(tokens):
    
    return (
        [START_TOKEN]
        + list(tokens)
        + [END_TOKEN]
    )


def replace_unknown_words(tokens):
    
    return [
        word
        if word in vocab
        else UNK_TOKEN
        for word in tokens
    ]

In [16]:
def unigram_probability_add_k(
    word,
    k=K
):
    count = unigram_counts.get(
        word,
        0
    )
    
    return (
        count + k
    ) / (
        metadata['total_tokens'] + k * V
    )

In [17]:
def bigram_probability_add_k(
    w1,
    w2,
    k=K
):
    count = bigram_counts.get(
        (w1, w2),
        0
    )
    
    context_count = bigram_context_counts.get(
        w1,
        0
    )
    
    return (
        count + k
    ) / (
        context_count + k * V
    )

In [18]:
def trigram_probability_add_k(
    w1,
    w2,
    w3,
    k=K
):
    count = trigram_counts.get(
        (w1, w2, w3),
        0
    )
    
    context_count = trigram_context_counts.get(
        (w1, w2),
        0
    )
    
    return (
        count + k
    ) / (
        context_count + k * V
    )

In [19]:
def quadrigram_probability_add_k(
    w1,
    w2,
    w3,
    w4,
    k=K
):
    count = quadrigram_counts.get(
        (w1, w2, w3, w4),
        0
    )
    
    context_count = quadrigram_context_counts.get(
        (w1, w2, w3),
        0
    )
    
    return (
        count + k
    ) / (
        context_count + k * V
    )

### Unigram Evaluation

In [20]:
def evaluate_unigram(dataset, k=K):
    total_log_probability = 0.0
    total_tokens_evaluated = 0

    for tokens in dataset["words"]:
        # Replace OOV words
        tokens = replace_unknown_words(tokens)

        # Add sentence boundaries
        sentence = add_sentence_tokens(tokens)

        # Evaluate every token
        for word in sentence:
            probability = unigram_probability_add_k(
                word, k
            )

            total_log_probability += log(probability)

            total_tokens_evaluated += 1

    perplexity = exp(
        -total_log_probability / total_tokens_evaluated
    )

    return perplexity, total_tokens_evaluated

In [21]:
unigram_dev_ppl, unigram_dev_tokens = evaluate_unigram(
    dev_df
)

print("Unigram Development Perplexity:", unigram_dev_ppl)
print("Tokens evaluated:", unigram_dev_tokens)

Unigram Development Perplexity: 1224.3586891991918
Tokens evaluated: 23683


In [22]:
unigram_test_ppl, unigram_test_tokens = evaluate_unigram(
    test_df
)

print("Unigram Test Perplexity:", unigram_test_ppl)
print("Tokens evaluated:", unigram_test_tokens)

Unigram Test Perplexity: 1225.4232796645538
Tokens evaluated: 23264


### Bigram Evaluation

In [23]:
def evaluate_bigram(dataset, k=K):
    total_log_probability = 0.0
    total_tokens_evaluated = 0

    for tokens in dataset["words"]:
        # Replace OOV words
        tokens = replace_unknown_words(tokens)

        # Add sentence boundaries
        sentence = add_sentence_tokens(tokens)

        # Predict every word from previous word
        for i in range(1, len(sentence)):

            w1 = sentence[i - 1]
            w2 = sentence[i]

            probability = bigram_probability_add_k(
                w1,
                w2,
                k
            )

            total_log_probability += log(probability)

            total_tokens_evaluated += 1

    perplexity = exp(
        -total_log_probability / total_tokens_evaluated
    )

    return perplexity, total_tokens_evaluated

In [24]:
bigram_dev_ppl, bigram_dev_tokens = evaluate_bigram(
    dev_df
)

print("Bigram Development Perplexity:", bigram_dev_ppl)
print("Tokens evaluated:", bigram_dev_tokens)

Bigram Development Perplexity: 970.3650830232621
Tokens evaluated: 22683


In [25]:
bigram_test_ppl, bigram_test_tokens = evaluate_bigram(
    test_df
)

print("Bigram Test Perplexity:", bigram_test_ppl)
print("Tokens evaluated:", bigram_test_tokens)

Bigram Test Perplexity: 933.7228441919145
Tokens evaluated: 22264


### Trigram Evaluation

In [26]:
def evaluate_trigram(dataset, k=K):
    total_log_probability = 0.0
    total_tokens_evaluated = 0

    for tokens in dataset["words"]:
        # Replace OOV words
        tokens = replace_unknown_words(tokens)

        # Add sentence boundaries
        sentence = add_sentence_tokens(tokens)

        # Predict from two previous tokens
        for i in range(2, len(sentence)):
            w1 = sentence[i - 2]
            w2 = sentence[i - 1]
            w3 = sentence[i]

            probability = trigram_probability_add_k(
                w1,
                w2,
                w3,
                k
            )

            total_log_probability += log(probability)

            total_tokens_evaluated += 1

    perplexity = exp(
        -total_log_probability / total_tokens_evaluated
    )

    return perplexity, total_tokens_evaluated

In [27]:
trigram_dev_ppl, trigram_dev_tokens = evaluate_trigram(
    dev_df
)

print("Trigram Development Perplexity:", trigram_dev_ppl)
print("Tokens evaluated:", trigram_dev_tokens)

Trigram Development Perplexity: 8999.061195489003
Tokens evaluated: 21683


In [28]:
trigram_test_ppl, trigram_test_tokens = evaluate_trigram(
    test_df
)

print("Trigram Test Perplexity:", trigram_test_ppl)
print("Tokens evaluated:", trigram_test_tokens)

Trigram Test Perplexity: 8410.037271904972
Tokens evaluated: 21264


### Quadrigram Evaluation

In [29]:
def evaluate_quadrigram(dataset, k=K):
    total_log_probability = 0.0
    total_tokens_evaluated = 0

    for tokens in dataset["words"]:
        # Replace OOV words
        tokens = replace_unknown_words(tokens)

        # Add sentence boundaries
        sentence = add_sentence_tokens(tokens)

        # Predict from three previous tokens
        for i in range(3, len(sentence)):
            w1 = sentence[i - 3]
            w2 = sentence[i - 2]
            w3 = sentence[i - 1]
            w4 = sentence[i]

            probability = quadrigram_probability_add_k(
                w1,
                w2,
                w3,
                w4,
                k
            )

            total_log_probability += log(probability)

            total_tokens_evaluated += 1

    perplexity = exp(
        -total_log_probability / total_tokens_evaluated
    )

    return perplexity, total_tokens_evaluated

In [30]:
quadrigram_dev_ppl, quadrigram_dev_tokens = evaluate_quadrigram(
    dev_df
)

print("Quadrigram Development Perplexity:", quadrigram_dev_ppl)
print("Tokens evaluated:", quadrigram_dev_tokens)

Quadrigram Development Perplexity: 40609.6493412221
Tokens evaluated: 20684


In [31]:
quadrigram_test_ppl, quadrigram_test_tokens = evaluate_quadrigram(
    test_df
)

print("Quadrigram Test Perplexity:", quadrigram_test_ppl)
print("Tokens evaluated:", quadrigram_test_tokens)

Quadrigram Test Perplexity: 38633.909550923694
Tokens evaluated: 20267


### Final result table

In [32]:
results = pd.DataFrame({
    "Model": [
        "Unigram",
        "Bigram",
        "Trigram",
        "Quadrigram"
    ],
    
    "Development Perplexity": [
        unigram_dev_ppl,
        bigram_dev_ppl,
        trigram_dev_ppl,
        quadrigram_dev_ppl
    ],
    
    "Test Perplexity": [
        unigram_test_ppl,
        bigram_test_ppl,
        trigram_test_ppl,
        quadrigram_test_ppl
    ],
    
    "K": [
        K,
        K,
        K,
        K
    ]
})

results

,Model,Development Perplexity,Test Perplexity,K
0,Unigram,1224.358689,1225.423280,0.3
1,Bigram,970.365083,933.722844,0.3
2,Trigram,8999.061195,8410.037272,0.3
3,Quadrigram,40609.649341,38633.909551,0.3
